In [ ]:
# environment

import os
import warnings
import tensorflow as tf 
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
import numpy as np

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0  #normalizing
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


introduce model

In [3]:
# normal neural network

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])


define loss function and optimizer

In [ ]:


loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam()


custom training loop

In [ ]:
# Set the number of times the full training data will be passed through the model.
epochs = 2

# This line is commented out; if used, it would repeat the dataset for the given number of epochs.
# train_dataset = train_dataset.repeat(epochs)

# Create a TensorFlow dataset from the training features and labels, then group samples into batches of 32.
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

# Loop over the dataset once for each epoch.
for epoch in range(epochs):
    # Print which epoch is starting.
    print(f'Start of epoch {epoch + 1}')

    # Loop through the training dataset batch by batch, and also keep track of the batch number.
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        # Open a GradientTape context so TensorFlow can record operations for automatic differentiation.
        with tf.GradientTape() as tape:
            # Run the model on the current input batch in training mode to get predictions.
            logits = model(x_batch_train, training=True)  # Forward pass

            # Compute the loss by comparing the true labels with the model predictions.
            loss_value = loss_fn(y_batch_train, logits)  # Compute loss

        # Compute gradients of the loss with respect to the model's trainable parameters.
        grads = tape.gradient(loss_value, model.trainable_weights)

        # Update the model's trainable weights using the optimizer and the computed gradients.
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        # Check every 200 steps whether it is time to print the current loss.
        if step % 200 == 0:
            # Print the current epoch, step number, and loss value for monitoring training progress.
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')

Start of epoch 1
Epoch 1 Step 0: Loss = 2.3494479656219482
Epoch 1 Step 200: Loss = 0.41628581285476685
Epoch 1 Step 400: Loss = 0.1846248209476471
Epoch 1 Step 600: Loss = 0.17040826380252838
Epoch 1 Step 800: Loss = 0.1742211878299713
Epoch 1 Step 1000: Loss = 0.4213247299194336
Epoch 1 Step 1200: Loss = 0.18662497401237488
Epoch 1 Step 1400: Loss = 0.2659141719341278
Epoch 1 Step 1600: Loss = 0.2439383566379547
Epoch 1 Step 1800: Loss = 0.1940375417470932
Start of epoch 2
Epoch 2 Step 0: Loss = 0.1151578277349472
Epoch 2 Step 200: Loss = 0.12673912942409515
Epoch 2 Step 400: Loss = 0.13386085629463196
Epoch 2 Step 600: Loss = 0.0434107668697834
Epoch 2 Step 800: Loss = 0.08645936101675034
Epoch 2 Step 1000: Loss = 0.27402257919311523
Epoch 2 Step 1200: Loss = 0.10767773538827896
Epoch 2 Step 1400: Loss = 0.18032266199588776
Epoch 2 Step 1600: Loss = 0.18492206931114197
Epoch 2 Step 1800: Loss = 0.0938320979475975


 ### Part two: adding an accuracy metric to monitor model performance

In [ ]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0 

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


Define the model

In [7]:
# define the Model

model = Sequential([ 
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


defining loss function, optimization, and metric

In [8]:
# Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


creating custom training loop

In [ ]:
# implement the Custom Training Loop with Accuracy

epochs = 5  # Number of epochs for training

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')
    
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)
        
        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    
    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()


Start of epoch 1
Epoch 1 Step 0: Loss = 2.3971261978149414 Accuracy = 0.09375
Epoch 1 Step 200: Loss = 0.3925401568412781 Accuracy = 0.8277363181114197
Epoch 1 Step 400: Loss = 0.1705845147371292 Accuracy = 0.8647911548614502
Epoch 1 Step 600: Loss = 0.19319432973861694 Accuracy = 0.8811876177787781
Epoch 1 Step 800: Loss = 0.15377020835876465 Accuracy = 0.894389808177948
Epoch 1 Step 1000: Loss = 0.4779375195503235 Accuracy = 0.9018481373786926
Epoch 1 Step 1200: Loss = 0.16090728342533112 Accuracy = 0.9086177945137024
Epoch 1 Step 1400: Loss = 0.22540731728076935 Accuracy = 0.9133431315422058
Epoch 1 Step 1600: Loss = 0.22848281264305115 Accuracy = 0.9166145920753479
Epoch 1 Step 1800: Loss = 0.188115194439888 Accuracy = 0.9204434752464294
Start of epoch 2
Epoch 2 Step 0: Loss = 0.09969979524612427 Accuracy = 1.0
Epoch 2 Step 200: Loss = 0.2113967090845108 Accuracy = 0.9586442708969116
Epoch 2 Step 400: Loss = 0.07432468980550766 Accuracy = 0.9578397870063782
Epoch 2 Step 600: Loss =

 ### Part three: creating custom callback to log additional metrics and information during training